# SentinelVision AI
## Notebook 03 — Feature Extraction

### Objective

Notebook 01 analyzed dataset metadata.

Notebook 02 inspected the real video content.

This notebook converts video clips into reusable numeric embeddings.

The goal is to create one feature vector per clip so later anomaly models can train without repeatedly decoding raw video.

The flow is:

`clip manifest → frame sampling → resize frames → frame features → clip embeddings → saved embedding table`

In [4]:
from pathlib import Path

import cv2
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch

from tqdm.auto import tqdm
from transformers import AutoImageProcessor, AutoModel

We are still not fine-tuning a model.

We are using VideoMAE as a frozen feature extractor.

That means:

`video clip → pretrained video model → embedding vector`

The anomaly models in Notebook 04 and Notebook 05 will train on these stronger embeddings.

In [5]:
seed = 42

np.random.seed(seed)
torch.manual_seed(seed)

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

device

device(type='cpu')

Load clip

In [6]:
metadata_dir = Path("../data/metadata")
embeddings_dir = Path("../data/embeddings")

embeddings_dir.mkdir(
    parents=True,
    exist_ok=True,
)

clip_manifest = pd.read_parquet(
    metadata_dir / "clip_manifest.parquet"
)

clip_manifest.shape

(44996, 12)

In [7]:
clip_manifest.head()

,clip_id,video_id,file_path,category,binary_label,split,clip_index,start_seconds,end_seconds,clip_duration_seconds,source_video_duration_seconds,source_fps
0,Abuse002_x264_clip_00000,Abuse002_x264,data\raw\ucf-crime\Anomaly-Videos-Part-1\Anoma...,Abuse,1,test,0,0.0,4.0,4.0,28.833,30.0
1,Abuse002_x264_clip_00001,Abuse002_x264,data\raw\ucf-crime\Anomaly-Videos-Part-1\Anoma...,Abuse,1,test,1,4.0,8.0,4.0,28.833,30.0
2,Abuse002_x264_clip_00002,Abuse002_x264,data\raw\ucf-crime\Anomaly-Videos-Part-1\Anoma...,Abuse,1,test,2,8.0,12.0,4.0,28.833,30.0
3,Abuse002_x264_clip_00003,Abuse002_x264,data\raw\ucf-crime\Anomaly-Videos-Part-1\Anoma...,Abuse,1,test,3,12.0,16.0,4.0,28.833,30.0
4,Abuse002_x264_clip_00004,Abuse002_x264,data\raw\ucf-crime\Anomaly-Videos-Part-1\Anoma...,Abuse,1,test,4,16.0,20.0,4.0,28.833,30.0


#### Resolve video paths

In [8]:
project_root = Path("..").resolve()


def resolve_video_path(file_path: str) -> Path:
    """
    Convert a stored video path into an absolute path.
    """
    path = Path(file_path)

    if path.is_absolute():
        return path

    return project_root / path

In [9]:
example_path = resolve_video_path(
    clip_manifest.iloc[0]["file_path"]
)

print(example_path)
print("Exists:", example_path.exists())

C:\Users\tevin\OneDrive\Desktop\Computer-Vision\SentinelVision-AI\data\raw\ucf-crime\Anomaly-Videos-Part-1\Anomaly-Videos-Part-1\Abuse\Abuse002_x264.mp4
Exists: True


#### Create a balanced VideoMAE extraction sample

Start here from the old Notebook 03.

We keep the same balanced sampling logic so Notebook 04 and Notebook 05 remain fair.

Start with 100 normal and 100 anomalous clips per split.

After this works, we can increase the sample size.

In [10]:
sample_per_label_per_split = 100

clip_sample = (
    clip_manifest
    .groupby(["split", "binary_label"], group_keys=False)
    .apply(
        lambda group: group.sample(
            n=min(len(group), sample_per_label_per_split),
            random_state=seed,
        )
    )
    .reset_index(drop=True)
)

print("Sample shape:", clip_sample.shape)
print()
print(clip_sample.groupby(["split", "binary_label"]).size())

Sample shape: (600, 12)

split       binary_label
test        0               100
            1               100
train       0               100
            1               100
validation  0               100
            1               100
dtype: int64


C:\Users\tevin\AppData\Local\Temp\ipykernel_12648\3328362982.py:6: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(


#### Frame sampling functions

VideoMAE expects a fixed number of frames.

For each 4-second clip, we sample 16 evenly spaced frames.

If a clip returns fewer frames, we repeat the last available frame so the model still receives 16 frames.

In [ ]:
def read_frame_at_time(
    video_path: str | Path,
    timestamp_seconds: float,
) -> np.ndarray | None:
    """
    Read one RGB frame from a video at a specific timestamp.
    """
    capture = cv2.VideoCapture(str(video_path))

    if not capture.isOpened():
        capture.release()
        return None

    capture.set(
        cv2.CAP_PROP_POS_MSEC,
        timestamp_seconds * 1000,
    )

    success, frame = capture.read()
    capture.release()

    if not success or frame is None:
        return None

    frame_rgb = cv2.cvtColor(
        frame,
        cv2.COLOR_BGR2RGB,
    )

    return frame_rgb

In [ ]:
def sample_clip_frames(
    video_path: str | Path,
    start_seconds: float,
    end_seconds: float,
    number_of_frames: int = 8,
) -> list[np.ndarray]:
    """
    Sample evenly spaced frames from one clip window.
    """
    if end_seconds <= start_seconds:
        return []

    timestamps = np.linspace(
        start_seconds,
        max(end_seconds - 0.05, start_seconds),
        number_of_frames,
    )

    frames = []

    for timestamp in timestamps:
        frame = read_frame_at_time(
            video_path=video_path,
            timestamp_seconds=float(timestamp),
        )

        if frame is not None:
            frames.append(frame)

    return frames

#### Load VideoMAE as a frozen feature extractor

We use the pretrained Hugging Face Model without fine-tuning.
The model produces stronger video-aware embeddings than simple RGB statistics.

#### Resize frames

In [ ]:
def resize_frame(
    frame: np.ndarray,
    size: tuple[int, int] = (112, 112),
) -> np.ndarray:
    """
    Resize one RGB frame to a fixed spatial size.
    """
    resized = cv2.resize(
        frame,
        size,
        interpolation=cv2.INTER_AREA,
    )

    return resized

In [ ]:
def preprocess_frames(
    frames: list[np.ndarray],
    size: tuple[int, int] = (112, 112),
) -> list[np.ndarray]:
    """
    Resize all frames in a clip.
    """
    return [
        resize_frame(frame, size=size)
        for frame in frames
    ]

#### Build simple frame features

In [ ]:
def frame_to_features(frame: np.ndarray) -> np.ndarray:
    """
    Convert one RGB frame into simple numeric features.

    Features:
    - mean RGB values
    - standard deviation RGB values
    - grayscale brightness mean
    - grayscale brightness standard deviation
    """
    frame_float = frame.astype(np.float32) / 255.0

    rgb_mean = frame_float.mean(axis=(0, 1))
    rgb_std = frame_float.std(axis=(0, 1))

    gray = cv2.cvtColor(
        frame,
        cv2.COLOR_RGB2GRAY,
    ).astype(np.float32) / 255.0

    gray_mean = np.array([gray.mean()])
    gray_std = np.array([gray.std()])

    features = np.concatenate(
        [
            rgb_mean,
            rgb_std,
            gray_mean,
            gray_std,
        ]
    )

    return features

#### Convert clip to embedding

In [ ]:
def clip_to_embedding(
    video_path: str | Path,
    start_seconds: float,
    end_seconds: float,
    number_of_frames: int = 8,
    frame_size: tuple[int, int] = (112, 112),
) -> np.ndarray | None:
    """
    Convert one clip window into one numeric embedding.
    """
    frames = sample_clip_frames(
        video_path=video_path,
        start_seconds=start_seconds,
        end_seconds=end_seconds,
        number_of_frames=number_of_frames,
    )

    if not frames:
        return None

    processed_frames = preprocess_frames(
        frames=frames,
        size=frame_size,
    )

    frame_features = np.array(
        [
            frame_to_features(frame)
            for frame in processed_frames
        ]
    )

    # Aggregate frame-level features into one clip-level vector.
    clip_mean = frame_features.mean(axis=0)
    clip_std = frame_features.std(axis=0)

    clip_embedding = np.concatenate(
        [
            clip_mean,
            clip_std,
        ]
    )

    return clip_embedding

#### Test on one clip

In [ ]:
test_clip = clip_sample.iloc[0]

test_video_path = resolve_video_path(
    test_clip["file_path"]
)

test_embedding = clip_to_embedding(
    video_path=test_video_path,
    start_seconds=test_clip["start_seconds"],
    end_seconds=test_clip["end_seconds"],
)

print("Embedding:", test_embedding)
print("Shape:", test_embedding.shape if test_embedding is not None else None)

#### Extract embeddings for sample clips

In [ ]:
embedding_records = []

for row in tqdm(
    clip_sample.itertuples(index=False),
    total=len(clip_sample),
    desc="Extracting clip embeddings",
):
    video_path = resolve_video_path(row.file_path)

    embedding = clip_to_embedding(
        video_path=video_path,
        start_seconds=row.start_seconds,
        end_seconds=row.end_seconds,
        number_of_frames=8,
        frame_size=(112, 112),
    )

    if embedding is None:
        continue

    record = {
        "clip_id": row.clip_id,
        "video_id": row.video_id,
        "category": row.category,
        "binary_label": row.binary_label,
        "split": row.split,
        "start_seconds": row.start_seconds,
        "end_seconds": row.end_seconds,
    }

    for index, value in enumerate(embedding):
        record[f"feature_{index:03d}"] = float(value)

    embedding_records.append(record)

embeddings = pd.DataFrame(embedding_records)

embeddings.shape

In [ ]:
embeddings.head()

#### Check extraction coverage

In [ ]:
requested_clips = len(clip_sample)
extracted_clips = len(embeddings)

coverage = extracted_clips / requested_clips

print(f"Requested clips: {requested_clips:,}")
print(f"Extracted clips: {extracted_clips:,}")
print(f"Coverage: {coverage:.2%}")

#### Inspect embedding columns

In [ ]:
feature_columns = [
    column
    for column in embeddings.columns
    if column.startswith("feature_")
]

print("Number of feature columns:", len(feature_columns))
print(feature_columns[:10])

In [ ]:
embeddings[feature_columns].describe().T.head(20)

#### Save sample embeddings

In [ ]:
sample_embeddings_path = (
    embeddings_dir / "clip_embeddings_sample.parquet"
)

embeddings.to_parquet(
    sample_embeddings_path,
    index=False,
)

sample_embeddings_path

In [ ]:
saved = pd.read_parquet(sample_embeddings_path)

print(saved.shape)
print(saved.groupby(["split", "binary_label"]).size())

#### Quick visual check of embedding space

In [ ]:
plt.figure(figsize=(8, 6))

for label in sorted(embeddings["binary_label"].unique()):
    subset = embeddings[
        embeddings["binary_label"] == label
    ]

    plt.scatter(
        subset["feature_000"],
        subset["feature_001"],
        label=f"label={label}",
        alpha=0.6,
    )

plt.title("Simple Embedding Check")
plt.xlabel("feature_000")
plt.ylabel("feature_001")
plt.legend()
plt.tight_layout()
plt.show()

# Conclusion

Notebook 03 created the first reusable feature table for SentinelVision AI.

## What was done

We:

- loaded the clip manifest,
- sampled a manageable subset of clips,
- read frames from real video files,
- resized frames to a consistent shape,
- extracted simple numeric frame features,
- aggregated frame features into clip embeddings,
- saved the result as a Parquet file.

## Output created

`data/embeddings/clip_embeddings_sample.parquet`

## Why this matters

The project now has model-ready data.

Instead of decoding videos repeatedly, later notebooks can load the saved embedding table directly.

## Important limitation

These are simple statistical visual embeddings, not deep pretrained video embeddings.

They are good enough for building the first anomaly-detection baseline.

Later, this feature extraction stage can be upgraded to pretrained CNN or video-model embeddings.

## Next notebook

Notebook 04 will train baseline anomaly models:

`clip embeddings → distance threshold baseline → Isolation Forest → evaluation`